[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oceanmodeling/xarray-selafin/blob/main/workshop/02_advanced.ipynb)

*Run this notebook interactively in your browser via the Colab link above (no installation required).**

# Advanced: building a TELEMAC atmospheric forcing file from ERA5

This notebook shows a complete, real-world workflow with `xarray-selafin`:

1. Load **ERA5 reanalysis** (10 m wind + mean sea-level pressure) straight from
   Google's public ARCO Zarr archive — no download needed.
2. Open an existing **TELEMAC 2D mesh** stored as a Selafin (`.slf`) file.
3. **Interpolate** the gridded ERA5 fields onto the unstructured mesh nodes
   using barycentric (linear) weights.
4. Assemble a new `xarray.Dataset` with the right SELAFIN metadata and
   **write it as `global_atm.slf`**, ready to use as an atmospheric forcing
   file in a TELEMAC simulation.

!!! note "Prerequisites"
    - `xarray-selafin` installed (`pip install xarray-selafin`)
    - `gcsfs` (anonymous access to the ARCO ERA5 archive), `scipy`, `pandas`
    - ~1 GB of RAM for the interpolation step over the example period

## 1. Imports

`Delaunay` and `cKDTree` from SciPy do all the interpolation heavy lifting.

In [ ]:
# Colab bootstrap: install dependencies.
try:
    import google.colab  # noqa: F401
except ImportError:
    pass
else:
    !pip install -q --upgrade xarray-selafin gcsfs hvplot zarr dask panel geoviews cartopy

In [ ]:
import getpass
import os
import tempfile
from pathlib import Path

import xarray as xr
import pandas as pd
import numpy as np

import hvplot.pandas

from scipy.spatial import Delaunay
from scipy.spatial import cKDTree

### Working folders

Same convention as notebook 01:

- **`DATA`**: read-only sample files from the repo (`tests/data`). Not used here, kept for consistency.
- **`SCRATCH`**: your own writable folder — the mesh download and the generated `global_atm.slf` go there. Defaults to `$TMPDIR/xarray-selafin-workshop/$USER`; override with the `XSLF_SCRATCH` environment variable (on Colab this is a per-session folder).

In [ ]:
DATA = Path(os.environ.get("XSLF_DATA", "../tests/data")).resolve()
SCRATCH = Path(os.environ.get("XSLF_SCRATCH", Path(tempfile.gettempdir()) / "xarray-selafin-workshop" / getpass.getuser()))
SCRATCH.mkdir(parents=True, exist_ok=True)

print("scratch:", SCRATCH)

## 2. Interpolation helpers

We use **barycentric interpolation**: the regular ERA5 grid is triangulated
once with a Delaunay triangulation, and every mesh node is located inside a
triangle. The value at a node is then a weighted sum of the values at the 3
triangle vertices.

Two functions:

- `get_weights(in_xy, out_xy)` — computed **once**; returns the triangle
  vertices (`vert`) and weights (`wgts`) for each target node, plus fallback
  nearest-neighbour indices for nodes that fall *outside* the source grid
  (e.g. near the antimeridian or poles).
- `interp(values, ...)` — applied **per variable and per timestep**; it is a
  single vectorised `einsum`, so it is fast.

In [ ]:
def get_weights(in_xy, out_xy, d=2):
    """
    Triangulate an output mesh based on a set of input points and return the
    corresponding barycentric weights.

    Parameters
    ----------
    @param in_xy (np.ndarray): Nx2 array of input point coordinates.
    @param out_xy (np.ndarray): Nx2 array of output point coordinates.
    @param d (int): Dimension of the input points, by default 2.

    @returns (4-uple): (vert, wgts, out_idx_out, in_idx_out), with:
        vert (np.ndarray): Nx3 array of triangle vertices.
        wgts (np.ndarray): Nx3 array of barycentric weights.
        out_idx_out (np.ndarray): Boolean array indicating which points
            are outside the input mesh.
        in_idx_out (np.ndarray): Index array of the nearest input points
            for points outside the input mesh.

    """
    t = Delaunay(in_xy)  # triangulate output mesh
    s = t.find_simplex(out_xy)
    vert = np.take(t.simplices, np.maximum(s, 0), axis=0)  # Use max to avoid negative indices
    t_ = np.take(t.transform, np.maximum(s, 0), axis=0)
    delta = out_xy - t_[:, d]
    bary = np.einsum("njk,nk->nj", t_[:, :d, :], delta)
    wgts = np.hstack((bary, 1 - bary.sum(axis=1, keepdims=True)))
    # Points outside the out_xy
    out_idx_out = s < 0
    if np.any(out_idx_out):
        # For points outside, find nearest neighbors
        tree = cKDTree(in_xy)
        _, in_idx_out = tree.query(out_xy[out_idx_out])
    else:
        in_idx_out = None
    return vert, wgts, out_idx_out, in_idx_out


def interp(values, vtx, wts, out_idx_out, in_idx_out):
    """
    Interpolate values using barycentric coordinates and weights

    Parameters
    ----------
    @param values (np.ndarray): Array of values to be interpolated.
    @param vtx (np.ndarray): Array of vertex indices.
    @param wts (np.ndarray): Array of barycentric weights.
    @param out_idx_out (np.ndarray): Boolean array indicating which points
        are outside the input mesh.
    @param in_idx_out (np.ndarray): Index array of the nearest input points
        for points outside the input mesh.

    @return (np.ndarray): Interpolated values

    """
    res = np.einsum("nj,nj->n", np.take(values, vtx), wts)
    if in_idx_out is not None:
        res[out_idx_out] = values[in_idx_out]
    return res

## 3. Load the ERA5 reanalysis (ARCO on Google Cloud)

ERA5 is available as a cloud-optimised Zarr archive
(`gcp-public-data-arco-era5`). We open it **lazily** with anonymous access and
chunk along time so nothing is downloaded until we actually need the data.

!!! warning
    The `chunks={"time": 48}` argument avoids a warning about chunk alignment;
    feel free to adapt it to your network/machine.

In [ ]:
# download ECMWF forecast from Google Cloud Storage (GCS) - Single-level reanalysis (N320)
ds_reanalysis = xr.open_zarr(
    "gs://gcp-public-data-arco-era5/co/single-level-reanalysis.zarr-v2",
    consolidated=True,
    storage_options={"token": "anon"},
    chunks={"time": 48}
)

### Select variables and time window

We keep only what TELEMAC needs for atmospheric forcing:

| ERA5 variable | Meaning | SELAFIN role |
|---|---|---|
| `u10` | 10 m zonal wind | WIND ALONG X |
| `v10` | 10 m meridional wind | WIND ALONG Y |
| `msl` | mean sea-level pressure (Pa) | AIR PRESSURE |

Adjust `start` / `end` to your simulation period. Keep it short while testing —
ERA5 is hourly, so 10 days = 240 frames.

In [ ]:
vars = ["u10", "v10", "msl"]
start = "2026-07-01"
end = "2026-07-10"
subset = ds_reanalysis[vars].sel(time=slice(start, end))
subset

visualize the ERA5 N320 mesh

In [ ]:
era5_n320 = subset.isel(time=0).to_dataframe().reset_index() # load in first time step and convert to pandas dataframe (needed for geoviews)

In [ ]:
era5_n320.hvplot.points(
    "longitude", "latitude", c="msl", cmap="rainbow4", alpha=0.5,width=1200, height=800, title="ERA5 N320 MSL",geo=True, tiles="EsriImagery"
)

## 4. Get the TELEMAC mesh

We reuse an existing global 2D mesh (generated in this
[companion notebook](https://tomsail.github.io/static/mesh_tomawac.html)):

- `global.slf` — the Selafin file containing the mesh geometry (node
  coordinates `x`/`y` and connectivity `ikle2`)
- `global.cli` — the boundary-condition (`.cli`) file, needed later by TELEMAC
  itself (not used in this notebook)

Both files are hosted on
[this Google Drive folder](https://drive.google.com/drive/u/2/folders/1SMNkHymAeuRfKh27pwPf7TgdIlOhTBvI)
and can be fetched with `curl` into `SCRATCH`:

In [ ]:
!curl -L -o {SCRATCH}/global.cli "https://drive.usercontent.google.com/download?id=1Al6Hf0Fz-_GqWieM4jWjJoOh2rJ3zTke&export=download&confirm=t"
!curl -L -o {SCRATCH}/global.slf "https://drive.usercontent.google.com/download?id=1vccGVoskI9pq0Zq_mI_aJxTO7SMO1cOx&export=download&confirm=t"

### Open the mesh with `xarray-selafin`

Opening with `engine="selafin"` gives us the node coordinates as `x`/`y`
coordinates and the mesh metadata (connectivity, boundary table, ...) as
**dataset attributes** — we will copy these attributes onto the new forcing
dataset before writing.

!!! note
    The `params` attribute is patched here because the `NPTFR` entry stored in
    this particular file disagrees with the actual number of boundary nodes
    (`ipobo`). Values are: `(1, 0, 0, 0, 0, 0, 0, n_boundary_nodes, 0, 1)`.

In [ ]:
slf_mesh = xr.open_dataset(SCRATCH / "global.slf", engine="selafin")
slf_mesh.attrs['params'] = (1, 0, 0, 0, 0, 0, 0, int(slf_mesh.attrs["ipobo"].max()), 0, 1) # seems there was an error before 
slf_mesh.attrs

## 5. Prepare source and target coordinates

- `in_xy`: the ERA5 grid points (flattened `(longitude, latitude)` pairs)
- `out_xy`: the mesh nodes we interpolate **onto**

Both meshes are in geographic lon/lat degrees here, which is fine for a global
mesh. For regional meshes, make sure both coordinate sets use the **same CRS**
(and preferably a projected one in metres) before triangulating.

In [ ]:
in_xy = np.column_stack((subset.longitude.values, subset.latitude.values))
out_xy = np.column_stack((slf_mesh.x.values%360, slf_mesh.y.values)) # modulo 360 to ensure longitudes are in the range [0, 360] for interpolation

Quick sanity check on one ERA5 timestep before interpolating:

In [ ]:
subset["u10"].sel(time="2026-07-01T00:00:00")

### Compute the interpolation weights (once)

This is the expensive step (Delaunay triangulation + point location), but it
only depends on the geometry, not on time. So we do it a single time and
reuse the weights for every variable and timestep.

It takes 6sec to interpolate on this simple mesh, but time scales with the resolution of target mesh.

In [ ]:
wgts_info = get_weights(in_xy, out_xy)

## 6. Interpolate onto the mesh and build the SELAFIN dataset

In [ ]:
subset_loaded = subset.load()  # Load the data into memory to avoid repeated disk access (~1.5min)

In [ ]:
data = {}
for var in vars:
    data[var] = np.empty((len(subset.time), slf_mesh.sizes["node"]), dtype=np.float32)
    for it, t in enumerate(subset.time.values):
        data[var][it, :] = interp(subset_loaded[var].sel(time=t).values, *wgts_info)

slf_atm = xr.Dataset(
    coords={
        "time": subset.time.values,
        "x": ("node", slf_mesh.x.values),
        "y": ("node", slf_mesh.y.values),
    },
    data_vars={var: (("time", "node"), data[var]) for var in vars},
)
slf_atm

### Attach the SELAFIN metadata

Two things are **mandatory** before writing (see the
[API reference](https://oceanmodeling.github.io/xarray-selafin/api/)):

1. `ikle2` (connectivity), `ipobo`, `params`, ... — we simply **reuse the mesh
   attributes** wholesale, since the forcing lives on the same mesh.
2. `attrs["variables"]`: a dict mapping **each data variable name** to a
   `(long_name, unit)` tuple. Keys must match the dataset variable names
   exactly. Use standard TELEMAC names/units (full list in the
   [API tables](https://oceanmodeling.github.io/xarray-selafin/api/#variable-definitions))
   so the file is recognised by other tools.

`date_start` is the timestamp of the first frame, as a
`(year, month, day, hour, minute, second)` tuple — TELEMAC stores time as
seconds since this origin.

!!! tip "Variable names on re-reading"
    When you re-open the file, variables are **auto-detected** from their
    long names, so `u10`/`v10`/`msl` will come back under their canonical
    TELEMAC IDs `X`, `Y` and `P`. This is expected — the canonical IDs are
    what TELEMAC and other Selafin tools recognise.


In [ ]:
slf_atm.attrs = slf_mesh.attrs
slf_atm.attrs["variables"] = {
    "u10": ("WIND ALONG X", "M/S"),
    "v10": ("WIND ALONG Y", "M/S"),
    "msl": ("AIR PRESSURE", "PASCAL"),
}
t0 = pd.Timestamp(subset.time.values[0])
slf_atm.attrs["date_start"] = (t0.year, t0.month, t0.day, t0.hour, t0.minute, t0.second)
slf_atm.attrs["date_start"]

## 7. Write the SELAFIN file

Importing `SelafinAccessor` registers the `.selafin` accessor on every
dataset; `write()` then serialises header + all time frames.

You can now point your TELEMAC steering file at the `global_atm.slf` written in `SCRATCH`
(`ATMOSPHERIC DATA FILE` / `FICHIER DE DONNEES ATMOSPHERIQUES`).

In [ ]:
fout = SCRATCH / "global_atm.slf"
slf_atm.selafin.write(fout)
f"saved in {fout}"